In [1]:
%pip install rank-bm25 sentence-transformers faiss-cpu langchain langchain-groq numpy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.4 MB/s eta 0:00:00


In [2]:
import os
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
import faiss
from langchain_groq import ChatGroq

In [ ]:
os.environ['GROQ_API_KEY'] = 'GROQ_API_KEY'
llm = ChatGroq(
    model="llama-3.1-8b-instant",  # ← guaranteed working
    temperature=0
)

In [41]:
documents = [
    'PPO is a reinforcement learning algorithm.',
    'SAC is an off-policy RL algorithm.',
    'BM25 is a ranking function used in search engines.',
    'Cross-encoders evaluate query and document together.',
    'FAISS is used for efficient similarity search.',
    'Hybrid retrieval combines sparse and dense methods.'
]
chunks = documents

In [42]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = embedder.encode(chunks)
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [43]:
def baseline_retrieve(query, k=3):
    q_emb = embedder.encode([query])
    D, I = index.search(np.array(q_emb), k)
    return [chunks[i] for i in I[0]]

In [44]:
def hyde_query(query):
    prompt = f'Write a detailed answer for: {query}'
    return llm.invoke(prompt).content

In [45]:
def dense_retrieve_hyde(query, k=3):
    hypo = hyde_query(query)
    q_emb = embedder.encode([hypo])
    sims = cosine_similarity(q_emb, doc_embeddings)[0]
    ranked = np.argsort(sims)[::-1][:k]
    return ranked

In [46]:
tokenized_corpus = [doc.split() for doc in chunks]
bm25 = BM25Okapi(tokenized_corpus)
def bm25_retrieve(query, k=3):
    scores = bm25.get_scores(query.split())
    ranked = np.argsort(scores)[::-1][:k]
    return ranked

In [47]:
def rrf_fusion(bm25_idx, dense_idx, k=60):
    scores = {}
    for rank, idx in enumerate(bm25_idx):
        scores[idx] = scores.get(idx, 0) + 1/(k + rank)
    for rank, idx in enumerate(dense_idx):
        scores[idx] = scores.get(idx, 0) + 1/(k + rank)
    sorted_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [doc[0] for doc in sorted_docs]

In [48]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
def rerank(query, doc_indices):
    pairs = [[query, chunks[i]] for i in doc_indices]
    scores = cross_encoder.predict(pairs)
    ranked = sorted(zip(doc_indices, scores), key=lambda x: x[1], reverse=True)
    return [i[0] for i in ranked]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [49]:
def advanced_retrieve(query, k=3):
    bm25_idx = bm25_retrieve(query, k)
    dense_idx = dense_retrieve_hyde(query, k)
    fused = rrf_fusion(bm25_idx, dense_idx)
    reranked = rerank(query, fused)
    return [chunks[i] for i in reranked[:k]]

In [50]:
def generate_answer(query, docs):
    context = '\n'.join(docs)
    prompt = f'Answer using context:\n{context}\n\nQuestion: {query}'
    return llm.invoke(prompt).content

In [51]:
query = 'Explain PPO'
baseline = generate_answer(query, baseline_retrieve(query))
advanced = generate_answer(query, advanced_retrieve(query))
print('Baseline:', baseline)
print('\nAdvanced:', advanced)

Baseline: PPO stands for Proximal Policy Optimization, which is a type of reinforcement learning (RL) algorithm. It's an on-policy model-free algorithm that's used to train agents to make decisions in complex environments.

In PPO, the agent learns to make decisions by interacting with the environment and receiving rewards or penalties for its actions. The goal is to maximize the cumulative reward over time, which is known as the return.

PPO uses a policy-based approach, where the agent learns a policy (a mapping from states to actions) that maximizes the expected return. The policy is updated using a combination of the old policy and the new policy, which is learned by sampling experiences from the environment.

The key features of PPO include:

1. **On-policy**: PPO learns from experiences collected by the agent itself, rather than from a replay buffer.
2. **Model-free**: PPO does not require a model of the environment, unlike model-based RL algorithms.
3. **Policy-based**: PPO lear

## Observation
The advanced RAG system improves retrieval using HyDE, hybrid retrieval, and re-ranking, resulting in more accurate answers.